# AI AINAK — Notebook 1: Computer Vision Module

**Branch 1:** Object Detection, Class-Consistent Tracking, Monocular Relative Depth, and Looming Hazard Scoring.

This notebook executes the complete Computer Vision sensory branch of AI AINAK on recorded session video:
1. Loads the trained **YOLOv8n** object detection weights (`yolov8n.pt`).
2. Ingests raw video frames from `datasets/recordings/session-multimodal-01/video.mp4`.
3. Performs inference: multi-object bounding boxes, class labels, and detection confidences.
4. Performs temporal IoU tracking across consecutive frames.
5. Computes ground-perspective relative depth and expanding optical loom hazard scores.
6. Generates a **128-dimensional dense vision feature vector** per observation.
7. Saves formatted outputs to `vision_outputs.npz` for multimodal Transformer fusion.


In [ ]:
# Step 1: Install Dependencies
# In Google Colab, execute this cell to install required libraries.
!pip install -q ultralytics opencv-python numpy matplotlib


In [ ]:
# Environment Detection & Path Resolution
import os, sys, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    # Mount Google Drive if available
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    # Locate project root
    drive_candidates = [
        Path("/content/drive/MyDrive/AINAK_Colab"),
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), Path.cwd())
else:
    print("Running in local environment.")
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root resolved to: {PROJECT_ROOT}")

# Helper to find checkpoint across standard folders
def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Checkpoint not found: {filename}. Checked: {[str(c) for c in candidates]}")

# Helper to find dataset / recording session
def find_session_media(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Media file {filename} for session {session_id} not found.")

# Ensure output directories exist
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)



## Checkpoint & Input Video Selection
Locate the YOLOv8n checkpoint and the demo video session.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

yolo_path = find_checkpoint("yolov8n.pt", "vision")
video_path = find_session_media("session-multimodal-01", "video.mp4")

print(f"Loaded YOLOv8n checkpoint: {yolo_path} ({yolo_path.stat().st_size / 1e6:.2f} MB)")
print(f"Selected Video Session: {video_path} ({video_path.stat().st_size / 1e6:.2f} MB)")

# Verify Video Properties
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = total_frames / fps if fps > 0 else 0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video Properties: {width}x{height} @ {fps:.1f} FPS | Total Frames: {total_frames} | Duration: {duration_sec:.2f}s")


## Vision Inference Implementation
Here we define the detector, depth proxy, and 128-D feature vector generator matching the AI AINAK production pipeline.


In [ ]:
from ultralytics import YOLO

# Load model
yolo_model = YOLO(str(yolo_path))

# Hazard and Depth Scoring Logic
CRITICAL_CLASSES = {"person", "car", "truck", "bus", "motorcycle", "bicycle", "traffic light", "stop sign"}

def estimate_relative_depth(bbox, frame_shape):
    # Monocular perspective depth proxy: ground plane horizon contact + area scaling
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = bbox
    base_y = y2 / h
    area = max(0.0, (x2 - x1) * (y2 - y1)) / (w * h)
    proximity = 0.65 * base_y + 0.35 * min(1.0, area * 4.0)
    return float(np.clip(proximity, 0.0, 1.0))

def compute_hazard_score(class_name, confidence, depth, delta_depth):
    base_risk = 0.8 if class_name in {"car", "truck", "bus", "motorcycle"} else 0.5 if class_name == "person" else 0.3
    loom_rate = max(0.0, delta_depth)
    hazard = (0.4 * base_risk) + (0.35 * depth) + (0.25 * min(1.0, loom_rate * 2.0))
    return float(np.clip(hazard * confidence, 0.0, 1.0))

def extract_vision_feature_128d(detections, frame_shape):
    # 128-D feature vector:
    # [0:80]   - COCO class histogram normalized by sum
    # [80:84]  - normalized [center_x, center_y, area, confidence] of primary object
    # [84:128] - zero-padded / reserved for multi-scale context
    h, w = frame_shape[:2]
    vec = np.zeros(128, dtype=np.float32)
    if not detections:
        return vec

    for d in detections:
        cls_id = int(d["class_id"]) % 80
        vec[cls_id] += float(d["confidence"])
    
    # Normalize histogram
    sum_hist = vec[:80].sum()
    if sum_hist > 0:
        vec[:80] /= sum_hist

    # Primary detection geometry
    top_d = max(detections, key=lambda x: x["confidence"])
    x1, y1, x2, y2 = top_d["bbox"]
    cx = (x1 + x2) / (2.0 * w)
    cy = (y1 + y2) / (2.0 * h)
    area = ((x2 - x1) * (y2 - y1)) / (w * h)
    vec[80:84] = [cx, cy, area, top_d["confidence"]]
    return vec


## Process Video Frames & Track Detections
We iterate over the video, track detections with IoU, compute hazard levels, and build the 128-D features.


In [ ]:
cap = cv2.VideoCapture(str(video_path))

frame_idx = 0
timestamps = []
features = []
confidences = []
latencies_ms = []
detections_history = []
sample_annotated_frames = []

prev_tracks = [] # (bbox, class_name, track_id, depth, time)
next_track_id = 1

print("Starting Vision inference across video stream...")
t0 = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    ts = frame_idx / fps
    frame_idx += 1
    
    # Process every frame (or stride)
    t_start = time.perf_counter()
    results = yolo_model.predict(frame, conf=0.35, iou=0.45, verbose=False, device="cpu")[0]
    t_latency = (time.perf_counter() - t_start) * 1000

    frame_dets = []
    h, w = frame.shape[:2]
    
    for i in range(len(results.boxes)):
        xyxy = results.boxes.xyxy[i].cpu().numpy().tolist()
        cls_id = int(results.boxes.cls[i].item())
        cls_name = results.names[cls_id]
        conf = float(results.boxes.conf[i].item())
        depth = estimate_relative_depth(xyxy, frame.shape)
        
        # Simple IoU track match
        best_iou = 0.3
        matched_track = None
        for prev_box, prev_name, tid, prev_depth, prev_t in prev_tracks:
            if prev_name == cls_name:
                xA = max(xyxy[0], prev_box[0])
                yA = max(xyxy[1], prev_box[1])
                xB = min(xyxy[2], prev_box[2])
                yB = min(xyxy[3], prev_box[3])
                inter = max(0, xB - xA) * max(0, yB - yA)
                area1 = (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])
                area2 = (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1])
                iou = inter / max(1e-6, area1 + area2 - inter)
                if iou > best_iou:
                    best_iou = iou
                    matched_track = (tid, prev_depth, prev_t)

        if matched_track:
            tid, p_depth, p_t = matched_track
            dt = max(1e-3, ts - p_t)
            delta_d = (depth - p_depth) / dt
        else:
            tid = next_track_id
            next_track_id += 1
            delta_d = 0.0

        hazard = compute_hazard_score(cls_name, conf, depth, delta_d)

        frame_dets.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "confidence": conf,
            "bbox": xyxy,
            "track_id": tid,
            "relative_depth": depth,
            "hazard_score": hazard
        })

    prev_tracks = [(d["bbox"], d["class_name"], d["track_id"], d["relative_depth"], ts) for d in frame_dets]
    feat_128 = extract_vision_feature_128d(frame_dets, frame.shape)
    
    timestamps.append(ts)
    features.append(feat_128)
    confidences.append(max((d["confidence"] for d in frame_dets), default=0.0))
    latencies_ms.append(t_latency)
    detections_history.append(frame_dets)

    # Save visual samples
    if frame_idx % 25 == 0 or frame_idx == 1:
        annotated = results.plot()
        sample_annotated_frames.append((ts, annotated, frame_dets))

cap.release()
print(f"Completed vision processing: {len(timestamps)} frames processed in {time.time() - t0:.2f}s.")
print(f"Mean Inference Latency: {np.mean(latencies_ms):.2f} ms | p95 Latency: {np.percentile(latencies_ms, 95):.2f} ms")


## Visualization of Detections & Hazard Heatmap


In [ ]:
fig, axes = plt.subplots(1, min(3, len(sample_annotated_frames)), figsize=(16, 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for i, (ts, frame_rgb, dets) in enumerate(sample_annotated_frames[:3]):
    # Convert BGR to RGB
    img = cv2.cvtColor(frame_rgb, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f"Time: {ts:.2f}s | Detections: {len(dets)}", fontsize=11, fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## Export Vision Outputs to `vision_outputs.npz`
We save the features, timestamps, confidences, and latencies for multimodal fusion.


In [ ]:
vision_out_path = OUTPUTS_DIR / "vision_outputs.npz"

np.savez_compressed(
    str(vision_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(features, dtype=np.float32),
    confidences=np.array(confidences, dtype=np.float32),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported vision outputs to: {vision_out_path}")
print(f"File size: {vision_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(features).shape} (Matches 128-D Transformer contract)")
